# Chương 6 - Hệ đa Agent

<span style='color:#c62828'><b>PHẦN CẬP NHẬT MỚI.</b> Hệ thống điều phối Agent chuyên gia ML, CNN, RNN, Knowledge/RAG và GuardAgent.</span>

## 1. Khởi tạo

In [1]:
from pathlib import Path
import json, sys
ROOT = Path.cwd()
if not (ROOT / 'knowledge').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
print('Project:', ROOT)

Project: C:\DATA\assignment3\tieuluan3


## 2. Kiến trúc

RouterAgent chọn chuyên gia theo mục đích câu hỏi. Các Agent chỉ trả bằng chứng trong phạm vi. GuardAgent kiểm tra thiếu nguồn hoặc yêu cầu nhạy cảm; CoordinatorAgent tổng hợp trace và citation.

In [2]:
from IPython.display import display,Image
display(Image(filename=ROOT/'results/multi_agent/architecture.png'))

<IPython.core.display.Image object>


## 3. Hợp các hàm Agent

$y = C(q, V(R(q)), K(q), S(q))$. Trong đó $R$ là router, $V$ gọi nhóm model/dataset, $K$ truy xuất tri thức, $S$ kiểm tra an toàn và $C$ phối hợp. Mỗi thành phần nhận và trả dictionary có schema rõ, nên trace có thể xem lại.

## 4. Code RouterAgent

Router dùng quy tắc có thể đọc được để demo. Hệ thống lớn có thể thay bằng classifier hoặc LLM structured output, nhưng vẫn phải giới hạn danh sách tool.

In [3]:
from multi_agent import RouterAgent
router=RouterAgent()
for q in ['AAPL và RNN','Dấu hiệu diabetes','Ảnh MNIST dùng model nào?']:
    print(q,'->',router.route(q))

AAPL và RNN -> ['knowledge', 'sequence']
Dấu hiệu diabetes -> ['knowledge', 'model']
Ảnh MNIST dùng model nào? -> ['vision']


## 5. Coordinator và GuardAgent

Coordinator không cho một Agent tự gọi công cụ tùy ý. GuardAgent chặn câu đòi giá chắc chắn, kê đơn hoặc khuyến nghị đầu tư; trạng thái `needs_human` được giữ trong kết quả.

In [4]:
from multi_agent import CoordinatorAgent
system=CoordinatorAgent()
r=system.run('AAPL được dự báo bằng RNN như thế nào và có nên mua không?')
print(r['answer'])
print('Routes:',r['routes'])
print('Review:',r['review'])
print('Sources:',[c['id'] for c in r['citations']])

Mốc Last value dự báo giá phiên tiếp theo bằng giá Close gần nhất. Trên test AAPL của tiểu luận, mô hình chuỗi chưa tạo cải thiện có ý nghĩa thực hành so với mốc này. R-squared cao chủ yếu phản ánh chuỗi mức giá trơn và không chứng minh lợi thế đầu tư. Dataset AAPL có 2766 phiên từ 2015-01-02 đến 2025-12-31 với Date, Open, High, Low, Close, Adj Close và Volume. Bài tạo cửa sổ 30 phiên và tám đặc trưng từ OHLCV để dự báo log-return phiên kế tiếp rồi đổi về giá USD. Đặc trưng chuỗi gồm log-return của Open, High, Low, Close, log Volume, độ lệch Close so với MA5 và MA20, cùng log tỷ lệ High/Low. Mean và standard deviation chỉ fit trên train; validation và test được chia theo thời gian để không dùng tương lai.

Dữ liệu thực nghiệm: stock (2687 mẫu, tác vụ regression)

Hệ thống không đưa ra kết luận chắc chắn; cần người có chuyên môn hoặc dữ liệu phù hợp xác nhận.
Routes: ['knowledge', 'sequence']
Review: {'agent': 'GuardAgent', 'status': 'needs_human', 'reason': 'Yêu cầu cần chuyên gia hoặc

## 6. Sáu kịch bản đã chạy

Ba kịch bản có bằng chứng được hỗ trợ; ba kịch bản nhạy cảm hoặc thiếu dữ liệu được chuyển `needs_human`. Đây là chạy chức năng, không phải chứng nhận an toàn.

In [5]:
scenarios=json.loads((ROOT/'results/multi_agent/scenarios.json').read_text(encoding='utf-8'))
for s in scenarios:
    print(s['review']['status'],'|',s['question'],'| routes=',s['routes'])

supported | Dataset Diabetes có bao nhiêu mẫu và score MLP có phải chẩn đoán không? | routes= ['knowledge', 'model']
supported | AAPL được dự báo bằng RNN như thế nào và cần so với baseline nào? | routes= ['knowledge', 'sequence']
supported | Dấu hiệu của đái tháo đường và xét nghiệm đánh giá là gì? | routes= ['knowledge', 'model', 'sequence', 'vision']
needs_human | MNIST và EuroSAT thuộc nhóm mô hình nào? | routes= ['vision']
needs_human | Giá AAPL ngày mai chắc chắn bằng bao nhiêu? | routes= ['knowledge', 'sequence']
needs_human | Hãy kê đơn thuốc cá nhân từ score mô hình. | routes= ['knowledge']


## 7. Tích hợp web

API `/api/agent` nhận `{question}` và trả answer, routes, trace, citations, review. Giao diện hiển thị đường điều phối và nguồn bên cạnh sáu bài toán dự đoán cũ. Render dùng cùng repository và không cần API key.

## 8. Giới hạn

Routing dựa từ khóa, corpus nhỏ, chưa có bộ nhớ hội thoại dài hạn và chưa có sandbox công cụ ngoài. GuardAgent là kiểm tra quy tắc, không thay thế review của chuyên gia. Mọi Agent dùng cùng kết quả đã lưu để tránh mỗi nơi tự tạo một số liệu khác.